## **Static Context**

In [1]:
from dataclasses import dataclass
@dataclass
class ColorContext:
    favorite_color: str = "blue"
    latest_favorite_color: str = "Yellow"

In [2]:
from langchain.agents import create_agent
agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColorContext
)

In [3]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext()
)

In [4]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='7bb06aa9-70f4-4b50-88e2-37493c514a98'),
              AIMessage(content='I don’t know your favourite colour. I don’t have access to that unless you tell me. Want me to guess, or should we do a quick quick-color quiz to figure it out? If you want a fast guess, I’ll start with blue. Would you like to try guessing or take a short quiz?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1738, 'prompt_tokens': 12, 'total_tokens': 1750, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1664, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CrieUEYrACYFM20hSlLKXDHvbAxWg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run-

## **Accessing Context**

- context schema is immutable

In [5]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user."""

    return runtime.context.favorite_color

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the latest favourite colour of the user"""
    return runtime.context.latest_favorite_color

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColorContext
)

In [7]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext()
)

/home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=ColorContext(favorite_col...favorite_color='Yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


In [8]:
pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='21ab8de6-3e73-4932-950e-6411ca2798cd'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_9GUqu1Ft8KrgRcGIs9djAIZZ', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 277, 'prompt_tokens': 149, 'total_tokens': 426, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CrielqZqF1XkFUrRws8d0ln7EiJia', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b6495-81b8-7c50-9c5c-8c1a2dbbc395-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_9GUq

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext(favorite_color="black")
)

pprint(response)

/home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=ColorContext(favorite_col...favorite_color='Yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='1f6939b4-c7e7-4f98-aa51-4267d48da6b9'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_oi63nUIh2b8DULObJjPz4PG2', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 341, 'prompt_tokens': 149, 'total_tokens': 490, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CrierpftqVfOc6TP0XvYCmWyKxrwn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b6495-9d30-7020-9238-57f614df93f1-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_oi63

## **Write the state**

In [10]:
from langchain.agents import AgentState
class CustomState(AgentState):
    favourite_colour: str

In [11]:
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite color of the user in the state once they've revealed it."""

    return Command[tuple[()]](
        update={
            "favourite_colour": favourite_colour,
            "messages": [ToolMessage("Successfully Update favourite colour", tool_call_id=runtime.tool_call_id)]
        }
    )



In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [14]:
from langchain.messages import HumanMessage

response = agent.invoke(
    
        {"messages": [HumanMessage(content="My favourite colour is orange.")]},
        {"configurable": {"thread_id": "1"}}
    
)

response

{'messages': [HumanMessage(content='My favourite colour is orange.', additional_kwargs={}, response_metadata={}, id='0713ec6d-8a00-475d-8f7d-76b94cb5b364'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_VA9tmmYbiVFlTtDdJWEdFSeH', 'function': {'arguments': '{"favourite_colour":"orange"}', 'name': 'update_favourite_colour'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 142, 'total_tokens': 489, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CriozJ43k8g53tWY2Klm10mVXMHxG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b649f-31ff-79c3-bebc-cb1d33aa6609-0', tool_calls=[{'name': 'update_favourite_colour', 'args

## **Read State**

In [15]:
@tool
def read_favorite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state."
    

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favorite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [16]:
from langchain.messages import HumanMessage

response = agent.invoke(
    
        {"messages": [HumanMessage(content="My favourite colour is orange.")]},
        {"configurable": {"thread_id": "1"}}
    
)

response

{'messages': [HumanMessage(content='My favourite colour is orange.', additional_kwargs={}, response_metadata={}, id='e160d435-9c6b-4273-971a-c8fdecb88491'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ftvsPjRU5XiG9yuwvXWg3Fv5', 'function': {'arguments': '{"favourite_colour":"orange"}', 'name': 'update_favourite_colour'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 163, 'total_tokens': 382, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CriuXktDEStfe4bPyoblcWEYQOFdg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b64a4-7050-7930-a219-f6e0188542e5-0', tool_calls=[{'name': 'update_favourite_colour', 'args

In [17]:
from langchain.messages import HumanMessage

response = agent.invoke(
    
        {"messages": [HumanMessage(content="What is my favorite color?")]},
        {"configurable": {"thread_id": "1"}}
    
)

response

{'messages': [HumanMessage(content='My favourite colour is orange.', additional_kwargs={}, response_metadata={}, id='e160d435-9c6b-4273-971a-c8fdecb88491'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ftvsPjRU5XiG9yuwvXWg3Fv5', 'function': {'arguments': '{"favourite_colour":"orange"}', 'name': 'update_favourite_colour'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 163, 'total_tokens': 382, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CriuXktDEStfe4bPyoblcWEYQOFdg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b64a4-7050-7930-a219-f6e0188542e5-0', tool_calls=[{'name': 'update_favourite_colour', 'args